In [8]:
import re
#padrão regex para o arquivo que vem do apache
format_pat = re.compile(
    r"(?P<host>[\d\.]+)\s"
    r"(?P<identity>\S*)\s"
    r"(?P<user>\S*)\s"
    r"\[(?P<time>.*?)\]\s"
    r'"(?P<request>.*?)"\s'
    r"(?P<status>\d+)\s"
    r"(?P<bytes>\S*)\s"
    r'"(?P<referer>.*?)"\s'
    r'"(?P<user_agent>.*?)"\s*'
)

In [9]:
logPath = "access_log.txt" #arquivo criado com IA já que o original não está disponível

In [10]:
URLCounts = {}
#aqui abrimos o arquivo e lê linha por linha, validando as coerentes
with open(logPath, "r") as f:
    for line in (l.rstrip() for l in f):
        match = format_pat.match(line)
        if match:
            access = match.groupdict()
            request = access['request']
            (action, URL, protocol) = request.split()
            # Alterado: has_key() não existe no Python 3, usa-se "in" - Se a URL existe no dicionário, passa contado, se não,
            if URL in URLCounts:
                URLCounts[URL] = URLCounts[URL] + 1
            else:
                URLCounts[URL] = 1

results = sorted(URLCounts, key=lambda i: int(URLCounts[i]), reverse=True)

for result in results[:20]:
    # Alterado: print virou função no Python 3
    print(result + ": " + str(URLCounts[result])) #aqui o objetivo é encontrar o erro, já que há lixo de memória no arquivo original

ValueError: not enough values to unpack (expected 3, got 1)

In [11]:
URLCounts = {}
#pra entender o que é que está dando erro, roda vasculhando o arquivo e observa as linhas que não tem apenas 3 colunas
with open(logPath, "r") as f:
    for line in (l.rstrip() for l in f):
        match = format_pat.match(line)
        if match:
            access = match.groupdict()
            request = access['request']
            fields = request.split()
            if (len(fields) != 3): #aqui*
                # Alterado: print virou função no Python 3
                print(fields)

['_\\xb0ZP\\x07tR\\xe5']
['\\x16\\x03\\x01\\x00\\xa5']


In [12]:
URLCounts = {}

with open(logPath, "r") as f:
    for line in (l.rstrip() for l in f):
        match = format_pat.match(line)
        if match:
            access = match.groupdict()
            request = access['request']
            fields = request.split()
            # Só conta requisições válidas, e joga fora as que não estão com 3 fields
            if (len(fields) == 3):
                URL = fields[1]
                if URL in URLCounts:
                    URLCounts[URL] = URLCounts[URL] + 1
                else:
                    URLCounts[URL] = 1

results = sorted(URLCounts, key=lambda i: int(URLCounts[i]), reverse=True)

for result in results[:20]:
    print(result + ": " + str(URLCounts[result]))

/blog/: 10
/search?q=machine+learning: 7
/blog/data-science-intro/: 4
/category/ml/: 3
/contact/: 3
/: 2
/blog/python-tips/: 2
/wp-login.php: 2
/robots.txt: 2
/favicon.ico: 2
/index.php: 2
/about/: 2
/xmlrpc.php: 2
/old-page-4.html: 1
/wp-content/themes/main/style.css: 1
/feed/: 1
/wp-content/uploads/chart.png: 1
/tag/regex/: 1


In [13]:
URLCounts = {}

with open(logPath, "r") as f:
    for line in (l.rstrip() for l in f):
        match = format_pat.match(line)
        if match:
            access = match.groupdict()
            request = access['request']
            fields = request.split()
            if (len(fields) == 3):
                (action, URL, protocol) = fields
                # Só conta requisições GET, ou seja, que o usuário de fato recebe, mas não altera no site, como o post que havia antes
                if (action == 'GET'):
                    if URL in URLCounts:
                        URLCounts[URL] = URLCounts[URL] + 1
                    else:
                        URLCounts[URL] = 1

results = sorted(URLCounts, key=lambda i: int(URLCounts[i]), reverse=True)

for result in results[:20]:
    print(result + ": " + str(URLCounts[result]))

/blog/: 10
/search?q=machine+learning: 7
/blog/data-science-intro/: 4
/category/ml/: 3
/contact/: 3
/: 2
/blog/python-tips/: 2
/robots.txt: 2
/favicon.ico: 2
/index.php: 2
/about/: 2
/xmlrpc.php: 2
/old-page-4.html: 1
/feed/: 1
/wp-content/uploads/chart.png: 1
/tag/regex/: 1


In [16]:
UserAgents = {}

with open(logPath, "r") as f:
    for line in (l.rstrip() for l in f):
        match = format_pat.match(line)
        if match:
            access = match.groupdict()
            agent = access['user_agent'] #a unica coisa que altera aqui é usar UserAgents pra verificar QUEM está fazendo a requisição
            if agent in UserAgents:
                UserAgents[agent] = UserAgents[agent] + 1
            else:
                UserAgents[agent] = 1

results = sorted(UserAgents, key=lambda i: int(UserAgents[i]), reverse=True)

for result in results:
    print(result + ": " + str(UserAgents[agent]))

Mozilla/5.0 (iPhone; CPU iPhone OS 14_2 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Mobile/15E148: 10
Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/14.0 Safari/605.1.15: 10
Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/87.0.4280.88 Safari/537.36: 10
Mozilla/5.0 (X11; Linux x86_64; rv:83.0) Gecko/20100101 Firefox/83.0: 10
Mozilla/5.0 (compatible; Googlebot/2.1; +http://www.google.com/bot.html): 10
Mozilla/5.0 (compatible; bingbot/2.0; +http://www.bing.com/bingbot.htm): 10
Mozilla/5.0 (compatible; AhrefsBot/7.0; +http://ahrefs.com/robot/): 10


In [17]:
URLCounts = {}

with open(logPath, "r") as f:
    for line in (l.rstrip() for l in f):
        match = format_pat.match(line)
        if match:
            access = match.groupdict()
            agent = access['user_agent']
            # Ignora bots, spiders, cache e agentes vazios ("-")
            if (not('bot' in agent or 'spider' in agent or
                    'Bot' in agent or 'Spider' in agent or
                    'W3 Total Cache' in agent or agent == '-')):
                request = access['request']
                fields = request.split()
                if (len(fields) == 3):
                    (action, URL, protocol) = fields
                    if (action == 'GET'):
                        if URL in URLCounts:
                            URLCounts[URL] = URLCounts[URL] + 1
                        else:
                            URLCounts[URL] = 1

results = sorted(URLCounts, key=lambda i: int(URLCounts[i]), reverse=True)

for result in results[:20]:
    print(result + ": " + str(URLCounts[result]))

/blog/: 9
/search?q=machine+learning: 5
/: 2
/contact/: 2
/blog/data-science-intro/: 2
/category/ml/: 2
/index.php: 2
/about/: 2
/feed/: 1
/wp-content/uploads/chart.png: 1
/xmlrpc.php: 1
/favicon.ico: 1
/tag/regex/: 1


In [18]:
URLCounts = {}

with open(logPath, "r") as f:
    for line in (l.rstrip() for l in f):
        match = format_pat.match(line)
        if match:
            access = match.groupdict()
            agent = access['user_agent']
            # Ignora bots, spiders, cache e agentes vazios novamente
            if (not('bot' in agent or 'spider' in agent or
                    'Bot' in agent or 'Spider' in agent or
                    'W3 Total Cache' in agent or agent == '-')):
                request = access['request']
                fields = request.split()
                if (len(fields) == 3):
                    (action, URL, protocol) = fields
                    # Só conta URLs terminadas em /, nao os arquivos
                    if (URL.endswith("/")):
                        if (action == 'GET'):
                            # Alterado: has_key() não existe no Python 3, usa-se "in"
                            if URL in URLCounts:
                                URLCounts[URL] = URLCounts[URL] + 1
                            else:
                                URLCounts[URL] = 1

results = sorted(URLCounts, key=lambda i: int(URLCounts[i]), reverse=True)

for result in results[:20]:
    # Alterado: print virou função no Python 3
    print(result + ": " + str(URLCounts[result]))

/blog/: 9
/: 2
/contact/: 2
/blog/data-science-intro/: 2
/category/ml/: 2
/about/: 2
/feed/: 1
/tag/regex/: 1
